# Neuronale Netze 11: Fashion-MNIST – Daten entdecken

## Was machen wir heute?

Fashion-MNIST ist eine Sammlung kleiner Bilder von Kleidungsstücken. Jedes Bild hat ein **Label**. Das Label sagt, zu welcher Klasse das Bild gehört.

Heute **trainieren wir kein Netz**. Wir lernen zuerst die Daten kennen:

1. Wie viele Bilder gibt es?
2. Wie sehen die zehn Klassen aus?
3. Was ist ein Pixel?
4. Wie oft kommt jede Klasse vor?
5. Welche Bilder lassen sich vielleicht schwer unterscheiden?

**So arbeitest du:** Öffne das Colabs. Führe die Codezellen von oben nach unten aus. Trage deine Antworten in das separate Leitprogramm ein. Ändere nur das ausdrücklich markierte Feld in E6.


## 1. Vorbereitung: Daten laden

Führe die nächste Zelle **einmal** aus. Beim ersten Mal lädt Colab die Bilder herunter. Das kann kurz dauern. Warte auf **«Die Daten sind bereit.»**

Der Datensatz hat **Trainingsbilder** und **Testbilder**. Heute sehen wir Trainingsbilder. Bei den Testbildern zählen wir nur, wie viele es sind.

Später lernt ein Netz mit den Trainingsbildern. Danach prüfen wir es mit den anderen Bildern: den Testbildern.


In [ ]:
#@title 1. Fashion-MNIST laden { display-mode: "form" }
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from torchvision.datasets import FashionMNIST

train_dataset = FashionMNIST(root="/content/fashion_mnist_data", train=True, download=True)
test_dataset = FashionMNIST(root="/content/fashion_mnist_data", train=False, download=True)

# Dieselben Klassen-IDs gelten für Training und Test.
class_names = [
    "T-Shirt/Oberteil", "Hose", "Pullover", "Kleid", "Mantel",
    "Sandale", "Hemd (Shirt)", "Turnschuh", "Tasche", "Stiefelette",
]

print("\nDie Daten sind bereit.")


## 2. Experiment E1: Umfang und Klassen

Führe die Zelle aus. Lies die Anzahl Bilder in beiden Teilen ab. Schau dir danach die **Klassen-ID** und den **Klassennamen** an.

Eine Klassen-ID ist eine Zahl von **0 bis 9**. Die Zahl 0 und die Zahl 6 meinen **nicht** dasselbe Kleidungsstück.


In [ ]:
#@title E1 – Daten und Klassen anzeigen { display-mode: "form" }
print(f"Training: {len(train_dataset):,} Bilder".replace(",", "'"))
print(f"Test: {len(test_dataset):,} Bilder".replace(",", "'"))
print(f"Klassen: {len(class_names)}")
print()
for class_id, class_name in enumerate(class_names):
    print(f"ID {class_id}: {class_name}")


## 3. Experiment E2: Ein Bild pro Klasse

Führe die Zelle aus. Du siehst **zehn Bilder**: aus jeder Klasse das erste Bild im Trainingsbestand.

Sieh zuerst nur auf die Bilder. Lies danach die Namen. Manche Formen erkennst du sofort. Bei anderen ist es schwieriger.


In [ ]:
#@title E2 – Zehn Beispielbilder anzeigen { display-mode: "form" }
train_labels = train_dataset.targets.numpy()
figure, axes = plt.subplots(2, 5, figsize=(12, 5), constrained_layout=True)

for class_id, axis in enumerate(axes.flat):
    first_index = int(np.flatnonzero(train_labels == class_id)[0])
    image = train_dataset.data[first_index].numpy()
    axis.imshow(image, cmap="gray", vmin=0, vmax=255)
    axis.set_title(f"{class_id}: {class_names[class_id]}", fontsize=10)
    axis.axis("off")

plt.show()


## 4. Experiment E3: Ein Bild besteht aus Pixeln

Führe die Zelle aus. Links siehst du ein ganzes Bild. Rechts siehst du einen **vergrösserten Ausschnitt** aus demselben Bild.

Ein **Pixel** ist ein kleines Bildfeld. Jedes Bild hat **28 Zeilen und 28 Spalten**. Der Pixelwert zeigt die Helligkeit: **0 ist schwarz**, **255 ist weiss**. Dazwischen liegen Grautöne.


In [ ]:
#@title E3 – Bild und Pixel untersuchen { display-mode: "form" }
pixel_image = train_dataset.data[0].numpy()
pixel_class_id = int(train_labels[0])

print(f"Bild 0 gehört zur Klasse: {class_names[pixel_class_id]} (ID {pixel_class_id})")
print(f"Bildgrösse: {pixel_image.shape[0]} Zeilen × {pixel_image.shape[1]} Spalten")
print("Mögliche Pixelwerte: 0 (schwarz) bis 255 (weiss)")
print(f"Beispiel: Pixel in Zeile 14, Spalte 14 hat den Wert {pixel_image[14, 14]}.")

figure, axes = plt.subplots(1, 2, figsize=(8, 4), constrained_layout=True)
axes[0].imshow(pixel_image, cmap="gray", vmin=0, vmax=255, interpolation="nearest")
axes[0].set_title("Ganzes Bild (28 × 28)")
axes[0].axis("off")

axes[1].imshow(pixel_image[10:18, 10:18], cmap="gray", vmin=0, vmax=255,
               interpolation="nearest")
axes[1].set_title("Ausschnitt: 8 × 8 Pixel")
axes[1].set_xticks(np.arange(-0.5, 8, 1), minor=True)
axes[1].set_yticks(np.arange(-0.5, 8, 1), minor=True)
axes[1].grid(which="minor", color="white", linewidth=0.6)
axes[1].tick_params(which="both", bottom=False, left=False,
                    labelbottom=False, labelleft=False)
plt.show()


## 5. Experiment E4: Wie oft kommt jede Klasse vor?

Führe die Zelle aus. Die Tabelle zählt die Bilder **für jede Klasse**. Lies zuerst die Zeilen für die IDs **0**, **6** und **9**. Lies dann die Zeile **Gesamt**.

Vergleiche: Haben alle Klassen gleich viele Bilder? Ein solcher Vergleich hilft später, Ergebnisse eines Netzes fair zu beurteilen.


In [ ]:
#@title E4 – Anzahl Bilder pro Klasse zählen { display-mode: "form" }
train_counts = np.bincount(train_labels, minlength=10)
test_counts = np.bincount(test_dataset.targets.numpy(), minlength=10)

count_table = pd.DataFrame({
    "ID": list(range(10)),
    "Klasse": class_names,
    "Training": train_counts,
    "Test": test_counts,
    "Zusammen": train_counts + test_counts,
})
total_row = pd.DataFrame([{
    "ID": "–", "Klasse": "Gesamt",
    "Training": int(train_counts.sum()),
    "Test": int(test_counts.sum()),
    "Zusammen": int(train_counts.sum() + test_counts.sum()),
}])
display(pd.concat([count_table, total_row], ignore_index=True))


## 6. Experiment E5: Ähnliche Kleidungsstücke vergleichen

Führe die Zelle aus. Du siehst **je drei Bilder** von **Hemd (Shirt)**, **Pullover** und **Mantel**. Jede Zeile zeigt eine Klasse.

Suche **ein gemeinsames Merkmal** und **einen möglichen Unterschied**. Achte zum Beispiel auf Ärmel, Länge und Umriss. Beschreibe nur, was du **in diesen Bildern** siehst. Ob ein späteres Netz die Klassen wirklich verwechselt, wissen wir heute noch nicht.


In [ ]:
#@title E5 – Hemd, Pullover und Mantel vergleichen { display-mode: "form" }
comparison_ids = [6, 2, 4]  # Hemd (Shirt), Pullover, Mantel
figure, axes = plt.subplots(3, 3, figsize=(8, 8), constrained_layout=True)

for row, class_id in enumerate(comparison_ids):
    first_three = np.flatnonzero(train_labels == class_id)[:3]
    for column, image_index in enumerate(first_three):
        axis = axes[row, column]
        axis.imshow(train_dataset.data[int(image_index)].numpy(),
                    cmap="gray", vmin=0, vmax=255)
        axis.set_title(f"{class_names[class_id]} – Bild {column + 1}", fontsize=10)
        axis.axis("off")

plt.show()


## 7. Experiment E6: Zwei Bilder vergleichen

Im Feld **`image_index`** steht zuerst **42**. Führe die Zelle aus. Ändere danach **nur die Zahl 42 zu 43** und führe **dieselbe Zelle** erneut aus.

Notiere beide Bildnummern und Klassen. Vergleiche die Bilder: Was ist ähnlich, was ist verschieden?

**Wichtig:** Eine Bildnummer ist der Platz eines Bildes im Trainingsbestand. Sie ist **keine Klassen-ID**.


In [ ]:
#@title E6 – Zwei Bilder vergleichen { display-mode: "form" }
image_index = 42 #@param {type:"integer"}

if 0 <= image_index < len(train_dataset):
    selected_image = train_dataset.data[image_index].numpy()
    selected_class_id = int(train_labels[image_index])
    print(f"Bildnummer: {image_index}")
    print(f"Label: {class_names[selected_class_id]} (Klassen-ID {selected_class_id})")
    plt.figure(figsize=(4, 4))
    plt.imshow(selected_image, cmap="gray", vmin=0, vmax=255)
    plt.title(f"Bild {image_index}: {class_names[selected_class_id]}")
    plt.axis("off")
    plt.show()
else:
    print("Bitte eine ganze Bildnummer von 0 bis 59'999 eingeben.")


## 8. Schluss: Vermutungen festhalten

Trage deine Schlussantworten im **separaten Leitprogramm** ein:

- Welche zwei Klassen könnte ein Netz verwechseln und warum? Begründe deine Vermutung mit einem sichtbaren Merkmal.
- Was müssten wir später messen, um deine Vermutung zu prüfen?

Das war **Datenanalyse vor dem Training**. In der nächsten Einheit können wir aus diesen Daten ein erstes Netz mit PyTorch bauen.


Copyright 2026 [Markus Ineichen](mailto:markus.ineichen1@sluz.ch)

Code licence: [MIT License](https://mit-license.org/)

Text license: [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International](https://creativecommons.org/licenses/by-nc-sa/4.0/)